# Install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install openai groq python-dotenv

# Import libraries

In [ ]:
import os
from google.colab import files
import pandas as pd
import time
import json
import re  # For parsing verdict
from groq import Groq  
from dotenv import load_dotenv

# Select LLM_as_a_Judge

In [ ]:
'''
LLM-as-a-judge models selected from Groq API provider:
['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'llama-3.1-8b-instant']

Duplicate this notebook to run per LLM-as-a-judge
Adjust judge_model accordingly
'''
judge_model = 'openai/gpt-oss-20b'

# Set up paths

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
# If judge_model is 'llama-3.1-8b-instant', the judge_name will be 'tinyllama'
judge_name = extract_model_name(judge_model)
file_name = 'verbosity_llama'
# Adjust file path to match the target environment
file_path = f'drive/MyDrive/ViJudge/data/datasets_before_judgement/verbosity/{file_name}.csv'
judge_prompts_path = 'drive/MyDrive/ViJudge/prompts/judge_prompts.jsonl'
output_path = f'drive/MyDrive/ViJudge/data/verbosity_bias/{judge_name}_verbosity_bias_for_{file_name}.csv'
print(file_path)
print(output_path)

# Upload file .env to Colab

In [ ]:
uploaded_file = files.upload()

# Get API Key

In [ ]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found")

# Connect with Groq

In [ ]:
groq_client = Groq(
    api_key = groq_api_key,
)

# Preprocessing

In [ ]:
df = pd.read_csv(file_path)
print(f"Loaded dataframe with {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
df_filtered = df[pd.notna(df['verbosity_response_turn1'])]
print(f"\nFiltered to {len(df_filtered)} questions")
display(df_filtered.head())

# Load prompts

In [ ]:
# Load Judge Prompts
judge_prompts = {}
with open(judge_prompts_path, 'r') as f:
    for line in f:
        prompt_data = json.loads(line)
        judge_prompts[prompt_data['name']] = prompt_data
print(f"Loaded prompts: {list(judge_prompts.keys())}")

# Generate judment

In [ ]:
def create_verbosity_configs(row):
    """
    Generate two configurations from a single row:
    1. Original-Repetitive: model A is original, model B is a verbose version of model A
    2. Repetitive-Original: inverse of configuration 1

    Returns:
    - List of 2 dicts, each representing one configuration
    """
    configs = []

    # Config 1: original-repetitive
    config_original_repetitive = row.copy()
    config_original_repetitive['response_turn1_b'] = row['verbosity_response_turn1']
    config_original_repetitive['verbosity_config'] = 'original-repetitive'
    configs.append(config_original_repetitive)

    # Config 2: repetitive-original
    config_repetitive_original = row.copy()
    config_repetitive_original['response_turn1_a'] = row['verbosity_response_turn1']
    config_repetitive_original['verbosity_config'] = 'repetitive-original'
    configs.append(config_repetitive_original)

    return configs

In [ ]:
# Function to Generate Judgement
# Only use turn_1 input since verbosity is applied to turn_1 only
def generate_judgement(client, row, judge_name='openai/gpt-oss-20b'):
    # Extract values, handle NaN
    question_1 = row['content_turn1'] if pd.notna(row['content_turn1']) else ''
    # question_2 = row['content_turn2'] if pd.notna(row['content_turn2']) else ''
    question_2 = ''
    ref_1 = row['reference_turn1'] if pd.notna(row['reference_turn1']) else ''
    # ref_2 = row['reference_turn2'] if pd.notna(row['reference_turn2']) else ''
    ref_2 = ''
    answer_1_a = row['response_turn1_a'] if pd.notna(row['response_turn1_a']) else ''
    # answer_2_a = row['response_turn2_a'] if pd.notna(row['response_turn2_a']) else ''
    answer_2_a = ''
    answer_1_b = row['response_turn1_b'] if pd.notna(row['response_turn1_b']) else ''
    # answer_2_b = row['response_turn2_b'] if pd.notna(row['response_turn2_b']) else ''
    answer_2_b = ''
    model_a = row['model_a'] if pd.notna(row['model_a']) else ''
    model_b = row['model_b'] if pd.notna(row['model_b']) else ''

    # Is multi-turn if has question_2
    multi_turn = bool(question_2)

    # Has reference if either ref_1 or ref_2 is not empty
    has_ref = bool(ref_1 or ref_2)

    # Is pair if has response b, else single
    judge_type = 'pair' if answer_1_b or answer_2_b else 'single'

    # Define prompt name
    if multi_turn:
        prompt_name = judge_type + '_multi_turn'
    else:
        prompt_name = judge_type + '_1_turn'

    if has_ref:
        prompt_name = prompt_name + '_reference'

    if prompt_name not in judge_prompts:
        raise ValueError(f'Prompt {prompt_name} not found')

    prompt_data = judge_prompts[prompt_name]
    system_prompt = prompt_data['system_prompt']
    template = prompt_data['prompt_template']

    # Format user_prompt
    if multi_turn:
        if judge_type == 'pair':
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_a_1=answer_1_a, answer_a_2=answer_2_a,
                answer_b_1=answer_1_b, answer_b_2=answer_2_b
            )
        else:  # single multi-turn
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_1=answer_1_a, answer_2=answer_2_a
            )
    else:
        # Single-turn: 
        if '{question_2}' in template:
            raise ValueError('Template not compatible with single-turn')
        if judge_type == 'pair':
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer_a=answer_1_a,
                answer_b=answer_1_b
            )
        else:
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer=answer_1_a
            )

    # Call API
    try:
        response = client.chat.completions.create(
            model=judge_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            max_tokens=2048
        )
        judgement = response.choices[0].message.content.strip()
    except Exception as e:
        error_message = str(e)
        if "Request too large" in error_message and "413" in error_message:
            print(f"Request too large for q_id below -> verdict = 'unknown'")
            judgement = 'unknown'
        else:
            raise e # Re-raise exceptions other than 'Request too large'

    verdict_match = re.search(r'\[\[(A|B|C|\d+)\]\]', judgement)
    verdict = verdict_match.group(1) if verdict_match else "Unknown"

    conversation_a = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_a, "role": "assistant"},
        # {"content": question_2, "role": "user"},
        # {"content": answer_2_a, "role": "assistant"},
    ]
    conversation_b = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_b, "role": "assistant"},
        # {"content": question_2, "role": "user"},
        # {"content": answer_2_b, "role": "assistant"},
    ]

    new_row = {
        'question_id': row['question_id'],
        'category': row['category'],
        'model_a': model_a,
        'model_b': model_b,
        'conversation_a': conversation_a,
        'conversation_b': conversation_b,
        'judge_name': judge_name,
        'judge_type': judge_type,
        'judgement_explanation': judgement,
        'verdict': verdict
    }

    return new_row

In [ ]:
def process_verbosity_bias_judgements(df, output_path, client, judge_model, n=2):
    """
    Evaluate verbosity bias by running 2 configurations

    Args:
    - df: input DataFrame loaded from CSV
    - output_path: path to save results CSV
    - question_list: list of target question IDs (integers)
    - client: API client for judge model calls
    - judge_model: judge model name
    - n: number of evaluation runs per configuration (default=2)

    Output: Writes CSV with columns:
    - question_id, category, model_a, model_b, conversation_a, conversation_b,
      judge_name, judge_type, judgement_explanation, verdict,
      verbosity_config (original-repetitive/repetitive-original), attempt (1,2,...,n)
    """

    # Load successfully processed question_ids
    existing_records = set()
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        for _, row in existing_df.iterrows():
            key = f"{row['question_id']}-{row['verbosity_config']}-{row['attempt']}"
            existing_records.add(key)
        print(f"Loaded {len(existing_records)} existing records from {output_path}")

    # List of configurations to run
    configs = []
    for _, row in df_filtered.iterrows():
        # Create 2 configurations
        verbosity_configs = create_verbosity_configs(row)

        # Each configuration runs n times
        for verbosity_config in verbosity_configs:
            for attempt in range(1, n + 1):
                configs.append({
                    'row': verbosity_config,
                    'verbosity_config': verbosity_config['verbosity_config'],
                    'attempt': attempt,
                    'question_id': row['question_id']
                })

    print(f"  - {len(df_filtered)} questions")
    print(f"  - 2 verbosity configs per question")
    print(f"  - {n} attempts per config")
    total_configs = len(df_filtered) * 2 * n
    print(f"\nTotal configurations to process: {len(df_filtered)} × 2 × {n} = {total_configs} total\n")

    # Process each configuration
    success_count = 0
    skip_count = 0
    error_count = 0

    for i, config in enumerate(configs, 1):
        row = config['row']
        verbosity_config = config['verbosity_config']
        attempt = config['attempt']
        q_id = config['question_id']

        # Create key to check if already processed
        record_key = f"{q_id}-{verbosity_config}-{attempt}"

        if record_key in existing_records:
            print(f"[{i}/{len(configs)}] ⏭️  Skipping {record_key} (already processed)")
            skip_count += 1
            continue

        try:
            print(f"[{i}/{len(configs)}] 🔄 Processing {record_key}...")

            # Call generate_judgement()
            judgement = generate_judgement(client, row, judge_model)

            # Add info about verbosity_config and attempt
            judgement['verbosity_config'] = verbosity_config
            judgement['attempt'] = attempt

            # Save to CSV (append)
            temp_df = pd.DataFrame([judgement])
            if os.path.exists(output_path):
                temp_df.to_csv(output_path, mode='a', header=False, index=False)
            else:
                temp_df.to_csv(output_path, mode='w', header=True, index=False)

            # Add to existing_records to avoid reprocessing on restart
            existing_records.add(record_key)

            success_count += 1
            print(f"  [{i}/{len(configs)}] ✅ Success: {record_key} | Verdict: {judgement['verdict']}")

            # Rate limit
            time.sleep(10)

        except Exception as e:
            error_count += 1
            print(f"[{i}/{len(configs)}] ❌ Error on {record_key}: {e}")
            continue

    print(f"\n{'='*50}")
    print(f"📊 Summary:")
    print(f"   ✅ Success: {success_count}")
    print(f"   ⏭️  Skipped: {skip_count}")
    print(f"   ❌ Errors: {error_count}")
    print(f"   📁 Results saved to: {output_path}")
    print(f"{'='*50}")

In [ ]:
# Run
process_verbosity_bias_judgements(
    df=df,
    output_path=output_path,
    client=groq_client,
    judge_model=judge_model,
    n=4
)

# Check

In [ ]:
result = pd.read_csv(output_path)
print(f"\nTotal results: {len(result)}")
print(f"\nVerbosity config distribution:")
print(result['verbosity_config'].value_counts())
print(f"\nAttempt distribution:")
print(result['attempt'].value_counts())
result.head(len(result))